In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import pandas as pd

# Import the CRUD Python module from Project One
from CRUD_Python_Module import AnimalShelter

# Configure JupyterDash for the hosted Codio/Jupyter environment
JupyterDash.infer_jupyter_proxy_config()


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "aacpass"

# Connect to MongoDB through the CRUD module
db = AnimalShelter(username, password)

# Retrieve all records for the initial/reset state
df = pd.DataFrame.from_records(db.read({}))

# Remove the MongoDB ObjectId column so Dash can display the data
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = "Grazioso Salvare Logo.png"

encoded_image = base64.b64encode(
    open(image_filename, "rb").read()
).decode()


app.layout = html.Div([

    html.Center([

        html.H1("Grazioso Salvare Rescue Dog Dashboard"),

        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image),
                style={"height": "180px"}
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),

        html.H3("Created by Aidan")
    ]),

    html.Hr(),

    html.Div([

        html.B("Interactive Filter Options:"),

        dcc.RadioItems(
            id="filter-type",

            options=[
                {
                    "label": "Water Rescue",
                    "value": "Water Rescue"
                },
                {
                    "label": "Mountain or Wilderness Rescue",
                    "value": "Mountain or Wilderness Rescue"
                },
                {
                    "label": "Disaster or Individual Tracking",
                    "value": "Disaster or Individual Tracking"
                },
                {
                    "label": "Reset",
                    "value": "Reset"
                }
            ],

            value="Reset",

            labelStyle={
                "display": "inline-block",
                "marginRight": "20px",
                "marginTop": "10px"
            }
        )
    ]),

    html.Hr(),

    dash_table.DataTable(

        id="datatable-id",

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict("records"),

        row_selectable="single",

        selected_rows=[
            0
        ] if len(df) > 0 else [],

        page_action="native",

        page_current=0,

        page_size=10,

        sort_action="native",

        sort_mode="multi",

        filter_action="native",

        column_selectable="single",

        style_table={
            "overflowX": "auto"
        },

        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "140px",
            "maxWidth": "220px",
            "whiteSpace": "normal"
        },

        style_header={
            "fontWeight": "bold"
        }
    ),

    html.Br(),

    html.Hr(),

    # Display the breed chart and map side-by-side
    html.Div(

        className="row",

        style={
            "display": "flex",
            "gap": "20px",
            "alignItems": "flex-start"
        },

        children=[

            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"width": "45%"}
            ),

            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"width": "55%"}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################


def get_rescue_query(filter_type):

    # Water Rescue
    if filter_type == "Water Rescue":

        return {

            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }


    # Mountain or Wilderness Rescue
    elif filter_type == "Mountain or Wilderness Rescue":

        return {

            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }


    # Disaster or Individual Tracking
    elif filter_type == "Disaster or Individual Tracking":

        return {

            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }


    # Reset returns all records
    return {}


####################################################
# Update the table when a rescue filter is selected
####################################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    query = get_rescue_query(filter_type)

    records = db.read(query)

    dff = pd.DataFrame.from_records(records)

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")


###############################################
# Update breed distribution visualization
###############################################

@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or "breed" not in dff.columns:
        return [
            html.P(
                "No matching animals are available for this filter."
            )
        ]

    # Count how many animals belong to each breed
    breed_counts = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
    )

    # Keep the chart readable by displaying the ten most common breeds
    # and combining all remaining breeds into an Other category
    if len(breed_counts) > 10:

        top_breeds = breed_counts.head(10)

        other_count = breed_counts.iloc[10:].sum()

        breed_counts = pd.concat([
            top_breeds,
            pd.Series({"Other": other_count})
        ])

    breed_counts = breed_counts.reset_index()

    breed_counts.columns = [
        "breed",
        "count"
    ]

    fig = px.pie(
        breed_counts,
        values="count",
        names="breed",
        title="Breed Distribution"
    )

    fig.update_traces(
        textposition="inside",
        textinfo="percent+label"
    )

    fig.update_layout(
        legend_title_text="Breed"
    )

    return [
        dcc.Graph(
            figure=fig
        )
    ]


###############################################
# Highlight a selected table column
###############################################

@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),

    [
        Input(
            "datatable-id",
            "selected_columns"
        )
    ]
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    return [

        {
            "if": {
                "column_id": i
            },

            "background_color": "#D2F3FF"
        }

        for i in selected_columns
    ]


###############################################
# Update map based on selected table row
###############################################

@app.callback(

    Output(
        "map-id",
        "children"
    ),

    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),

        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(viewData, index):

    if viewData is None:
        dff = df.copy()

    else:
        dff = pd.DataFrame.from_dict(viewData)


    if dff.empty:

        return [
            html.P(
                "No matching animal is available to display on the map."
            )
        ]


    # Default to the first row
    row = 0


    # Use the selected row when possible
    if index and index[0] < len(dff):
        row = index[0]


    latitude = dff.iloc[row]["location_lat"]

    longitude = dff.iloc[row]["location_long"]

    breed = dff.iloc[row]["breed"]

    animal_name = dff.iloc[row]["name"]


    # Make sure valid location information exists
    if pd.isna(latitude) or pd.isna(longitude):

        return [
            html.P(
                "Location information is not available for this animal."
            )
        ]


    if pd.isna(animal_name):
        animal_name = "Unnamed Animal"


    return [

        dl.Map(

            style={
                "width": "100%",
                "height": "500px"
            },

            center=[
                latitude,
                longitude
            ],

            zoom=10,

            children=[

                dl.TileLayer(
                    id="base-layer-id"
                ),

                dl.Marker(

                    position=[
                        latitude,
                        longitude
                    ],

                    children=[

                        dl.Tooltip(
                            str(breed)
                        ),

                        dl.Popup([

                            html.H3(
                                "Animal Name"
                            ),

                            html.P(
                                str(animal_name)
                            ),

                            html.P(
                                "Breed: {}".format(
                                    breed
                                )
                            )
                        ])
                    ]
                )
            ]
        )
    ]


##############################
# Run Dashboard
##############################

app.run_server(
    mode="external",
    debug=False,
    port=8052
)